# Different approaches to anomaly detection

This Python notebook proposes three different approaches for anomaly detection in network flows. In addition, their advantages and disadvantages are detailed.

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from river import anomaly
from river import preprocessing
from river import optim
from river import sketch
import re

In [75]:
# Training flows for the online learning algorithm
FLOWS_TRAIN_SCALER = 1000
FLOWS_TRAIN_OML = 100000


This dataset was created based on one of the datasets provided by Proactivanet. Subsequently, the following anomalies were introduced, with 2,500 flows of each type:

- Benign IP connecting to an anomalous IP during working hours
- Benign IP connecting to a benign IP at an anomalous time
- Benign IP connecting to an anomalous domain during working hours
- Benign IP connecting to an anomalous domain at an anomalous time
- Anomalous IP connecting to an anomalous IP during working hours

## Exact Match Detector

In [76]:
DATASET_PATH_DATASET2 = './dataset/'
DATASET_NAME_DATASET2 = 'dataset_anonymized.csv'

In [77]:
# Read CSV
dataset = pd.read_csv(DATASET_PATH_DATASET2 + DATASET_NAME_DATASET2, sep=',')

# Use the following features
dataset = dataset[['FIRST_SWITCHED', 'IPV_SRC_ADDR', 'L_SRC_PORT', 'IPV_DST_ADDR', 'DIRECTION', 'L_DST_PORT', 'IN_BYTES', 'OUT_BYTES', 'Label']]

# The FIRST_SWITCHED feature is converted into the hour of the day
dataset['FIRST_SWITCHED'] = pd.to_datetime(dataset['FIRST_SWITCHED']).dt.hour

# A PORT feature is created that is L_SRC_PORT if DIRECTION is 0 and L_DST_PORT if DIRECTION is 1
dataset['PORT'] = np.where(dataset['DIRECTION'] == 0, dataset['L_SRC_PORT'], dataset['L_DST_PORT'])

# Remove the columns L_SRC_PORT, L_DST_PORT, and DIRECTION
dataset = dataset.drop(columns=['L_SRC_PORT', 'L_DST_PORT', 'DIRECTION'])

# Preprocess IN_BYTES and OUT_BYTES by rounding to the nearest multiple of 100
dataset['IN_BYTES'] = dataset['IN_BYTES'].apply(lambda x: round(x, -2))
dataset['OUT_BYTES'] = dataset['OUT_BYTES'].apply(lambda x: round(x, -2))

# Remove label
dataset_sin_etiqueta = dataset.drop(columns=['Label'])


In [78]:
# Create the training set
X_train = dataset_sin_etiqueta[dataset['Label'] == 0].iloc[:100000]
print("Number of training samples: ", len(X_train))

Number of training samples:  100000


In [79]:
model = {}

for row in X_train.iterrows():
    key = tuple(row[1].values)
    model[key] = 0

El procedimiento anterior es igual al descrito anteriormente. Sin embargo, en este caso se ha entrenado con 200000 flujos benignos. En este caso, la fase de evaluación se ha realizado con 12500 flujos benignos y 12500 anómalos.

In [80]:
# Create the test set
X_test = dataset_sin_etiqueta[dataset['Label'] == 1].iloc[:12500]
X_test = pd.concat([X_test, dataset_sin_etiqueta[dataset['Label'] == 0].iloc[100000:112500]], ignore_index=True)

# Create the test set labels (1 for anomalies, 0 for normal flows)
y_test = np.ones(12500)
y_test = np.concatenate([y_test, np.zeros(12500)])

print("Number of anomalies in the test dataset: ", y_test[y_test == 1].shape[0])
print("Number of normal samples in the test dataset: ", y_test[y_test == 0].shape[0])


Number of anomalies in the test dataset:  12500
Number of normal samples in the test dataset:  12500


In [81]:
# Evaluation
tp = 0
tn = 0
fp = 0
fn = 0

for i, row in X_test.iterrows():

    key = tuple(row.values)
    is_anomaly = not key in model
    label = y_test[i]
    if is_anomaly and label == 1:
        tp += 1
    elif not is_anomaly and label == 0:
        tn += 1
    elif is_anomaly and label == 0:
        fp += 1
    elif not is_anomaly and label == 1:
        fn += 1

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
false_positive = fp / (fp + tn)
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)
print("False Positive Rate: ", false_positive)
print("F1 Score: ", f1_score)
print("TP: ", tp, "TN: ", tn, "FP: ", fp, "FN: ", fn)

Accuracy:  0.741
Precision:  0.6587615283267457
Recall:  1.0
False Positive Rate:  0.518
F1 Score:  0.7942811755361399
TP:  12500 TN:  6025 FP:  6475 FN:  0


All anomalies are correctly detected (recall = 1), but the number of false positives is very high. This is due to the model's inability to generalize, as it relies on exact match searches.

# Supervised approaches

## Supervised learning - DecisionTreeClassifier

In [82]:
# Shuffle the dataset
dataset = dataset.sample(frac=1, random_state=111).reset_index(drop=True)

# Create LabelEncoder objects
le_src = LabelEncoder()
le_dst = LabelEncoder()

# Fit and transform the IPV_SRC_ADDR and IPV_DST_ADDR features
dataset['IPV_SRC_ADDR'] = le_src.fit_transform(dataset['IPV_SRC_ADDR'])
dataset['IPV_DST_ADDR'] = le_dst.fit_transform(dataset['IPV_DST_ADDR'])

# Remove label column
dataset_sin_etiqueta = dataset.drop(columns=['Label'])


In this case, the model was trained with 100,000 benign flows and 6,250 anomalous flows.

In [83]:
# Split into training and test sets
X_train = dataset_sin_etiqueta[dataset['Label'] == 0].iloc[:93750]
# y_train contains 0s
y_train = np.zeros(93750)

# Add 6,250 anomalous flows to the training set
X_train = pd.concat([X_train, dataset_sin_etiqueta[dataset['Label'] == 1].iloc[:6250]])

# Add 1s to y_train for the anomalous samples
y_train = np.concatenate([y_train, np.ones(6250)])


In [84]:
print("Number of samples in the training dataset: ", y_train.shape[0])

Number of samples in the training dataset:  100000


In [85]:
# Create Grid Search for hyperparameter tuning
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Create the Decision Tree Classifier
clf = DecisionTreeClassifier(random_state=123)

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)
# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters: ", best_params)
# Create the Decision Tree Classifier with the best parameters
clf = DecisionTreeClassifier(**best_params, random_state=987)
# Fit the model
clf.fit(X_train, y_train)

Best parameters:  {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'entropy'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",987
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the cur

For evaluation, 6,250 benign flows and 6,250 anomalous flows will be used.

In [86]:
# Retrieve the anomalous flows from the dataset
X_test = dataset_sin_etiqueta[dataset['Label'] == 1].iloc[6250:]
y_test = np.ones(X_test.shape[0])

# Add 6,250 benign flows to the test set
X_test = pd.concat([X_test, dataset_sin_etiqueta[dataset['Label'] == 0].iloc[93750:100000]])
y_test = np.concatenate([y_test, np.zeros(6250)])


In [87]:
print("Number of samples in the test dataset: ", y_test.shape[0])

Number of samples in the test dataset:  12500


In [88]:
def obtain_metrics(y_test, y_pred):
    tp = np.sum((y_pred == 1) & (y_test == 1))
    tn = np.sum((y_pred == 0) & (y_test == 0))
    fp = np.sum((y_pred == 1) & (y_test == 0))
    fn = np.sum((y_pred == 0) & (y_test == 1))

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    # If there are no true positives, recall is 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    false_positive = fp / (fp + tn)
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print("Accuracy: ", accuracy)
    print("Precision: ", precision)
    print("Recall: ", recall)
    print("False Positive Rate: ", false_positive)
    print("F1 Score: ", f1_score)
    print("TP: ", tp, "TN: ", tn, "FP: ", fp, "FN: ", fn)

In [89]:
# Evaluate the classifier
y_pred = clf.predict(X_test)

obtain_metrics(y_test, y_pred)


Accuracy:  0.99976
Precision:  1.0
Recall:  0.99952
False Positive Rate:  0.0
F1 Score:  0.9997599423861727
TP:  6247 TN:  6250 FP:  0 FN:  3


In this case, the results are better than those obtained with the exact match search. Most anomalies are successfully detected, and the number of false positives is lower. However, it is important to note that a labeled dataset is required to train the model. Furthermore, it is very difficult for the model to generalize across different networks.

## Supervised learning - KNN

In [90]:
# Train KNN classifier
from sklearn.neighbors import KNeighborsClassifier
clf = KNeighborsClassifier()

param_grid = {
    'n_neighbors': [2, 3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree']
}

grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
print("Best parameters: ", best_params)

clf = KNeighborsClassifier(**best_params)
clf.fit(X_train, y_train)

Best parameters:  {'algorithm': 'auto', 'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'distance'}


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",3
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'manhattan'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [91]:
# Evaluate the classifier
y_pred = clf.predict(X_test)

obtain_metrics(y_test, y_pred)

Accuracy:  0.92528
Precision:  0.9975664545114189
Recall:  0.85264
False Positive Rate:  0.00208
F1 Score:  0.9194271911663217
TP:  5329 TN:  6237 FP:  13 FN:  921


## Supervised learning - Random Forest

In [92]:
# Train Random Forest classifier
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(random_state=123)
param_grid = {
    'n_estimators': [25, 50],
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
print("Best parameters: ", best_params)

clf = RandomForestClassifier(**best_params, random_state=123)
clf.fit(X_train, y_train)

Best parameters:  {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'entropy'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [93]:
# Evaluate the classifier
y_pred = clf.predict(X_test)

obtain_metrics(y_test, y_pred)

Accuracy:  0.99992
Precision:  1.0
Recall:  0.99984
False Positive Rate:  0.0
F1 Score:  0.999919993599488
TP:  6249 TN:  6250 FP:  0 FN:  1


## Supervised learning - Extra Trees

In [94]:
# Supervised learning - Extra Trees
from sklearn.ensemble import ExtraTreesClassifier
clf = ExtraTreesClassifier(random_state=123)
param_grid = {
    'n_estimators': [100, 200, 300],
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
print("Best parameters: ", best_params)

clf = ExtraTreesClassifier(**best_params, random_state=123)
clf.fit(X_train, y_train)

Best parameters:  {'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'entropy'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=FalseWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",False
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `m

In [95]:
# Evaluate the classifier
y_pred = clf.predict(X_test)

obtain_metrics(y_test, y_pred)

Accuracy:  0.99808
Precision:  1.0
Recall:  0.99616
False Positive Rate:  0.0
F1 Score:  0.9980763065084963
TP:  6226 TN:  6250 FP:  0 FN:  24


## Supervised learning - Logistic Regression

In [95]:
# Supervised learning - Logistic Regression
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
param_grid = [
    # Solvers que solo admiten L2 o sin penalización
    {
        'solver': ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag'],
        'penalty': ['l2', None],
        'C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'class_weight': [None, 'balanced'],
        'fit_intercept': [True, False],
        'tol': [1e-4, 1e-3]
    },
    
    # liblinear: solo l1 o l2
    {
        'solver': ['liblinear'],
        'penalty': ['l1', 'l2'],
        'C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'class_weight': [None, 'balanced'],
        'fit_intercept': [True, False],
        'tol': [1e-4, 1e-3]
    },
    
    # saga con l1
    {
        'solver': ['saga'],
        'penalty': ['l1'],
        'C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'class_weight': [None, 'balanced'],
        'fit_intercept': [True, False],
        'tol': [1e-4, 1e-3]
    },
    
    # saga con l2 o sin penalización
    {
        'solver': ['saga'],
        'penalty': ['l2', None],
        'C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'class_weight': [None, 'balanced'],
        'fit_intercept': [True, False],
        'tol': [1e-4, 1e-3]
    },
    
    # saga con elasticnet
    {
        'solver': ['saga'],
        'penalty': ['elasticnet'],
        'C': [0.01, 0.1, 1.0, 10.0, 100.0],
        'l1_ratio': [0.1, 0.5, 0.9],
        'class_weight': [None, 'balanced'],
        'fit_intercept': [True, False],
        'tol': [1e-4, 1e-3]
    }
]
grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1,verbose=1)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
print("Best parameters: ", best_params)

clf = LogisticRegression(**best_params, max_iter=1000)
clf.fit(X_train, y_train)

Fitting 3 folds for each of 640 candidates, totalling 1920 fits


/home/alberto/special-issue-cisis25/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/alberto/special-issue-cisis25/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/alberto/special-issue-cisis25/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated

Best parameters:  {'C': 0.01, 'class_weight': None, 'fit_intercept': True, 'penalty': 'l1', 'solver': 'liblinear', 'tol': 0.001}


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l1'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.01
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [96]:
# Evaluate the classifier
y_pred = clf.predict(X_test)

obtain_metrics(y_test, y_pred)

Accuracy:  0.87592
Precision:  0.9966180511519763
Recall:  0.7544
False Positive Rate:  0.00256
F1 Score:  0.858756033148165
TP:  4715 TN:  6234 FP:  16 FN:  1535


## Supervised learning - LSVC

In [97]:
## Supervised learning - LSVC
from sklearn.svm import LinearSVC
clf = LinearSVC(max_iter=10000, random_state=123)
param_grid = {
    'penalty': ['l2', 'l1'],
    'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'class_weight': [None, 'balanced'],
    'fit_intercept': [True, False],
    'tol': [1e-6, 1e-5, 1e-4]
}
grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
print("Best parameters: ", best_params)

clf = LinearSVC(**best_params, max_iter=40000, random_state=123)
clf.fit(X_train, y_train)

/home/alberto/special-issue-cisis25/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/alberto/special-issue-cisis25/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/alberto/special-issue-cisis25/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/alberto/special-issue-cisis25/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/alberto/special-issue-cisis25/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best parameters:  {'C': 10.0, 'class_weight': None, 'fit_intercept': True, 'penalty': 'l2', 'tol': 1e-05}


,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",1e-05
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [98]:
# Evaluate the classifier
y_pred = clf.predict(X_test)

obtain_metrics(y_test, y_pred)

Accuracy:  0.86088
Precision:  0.9982328252705986
Recall:  0.72304
False Positive Rate:  0.00128
F1 Score:  0.8386378398441124
TP:  4519 TN:  6242 FP:  8 FN:  1731


# Unsupervised approaches

In [17]:
# Create X_train and X_test for the unsupervised learning algorithms
X_train = dataset_sin_etiqueta[dataset['Label'] == 0].iloc[:100000]
X_test = dataset_sin_etiqueta[dataset['Label'] == 1].iloc[:12500]
X_test = pd.concat([X_test, dataset_sin_etiqueta[dataset['Label'] == 0].iloc[100000:112500]], ignore_index=True)
y_test = np.ones(12500)
y_test = np.concatenate([y_test, np.zeros(12500)])
y_train = np.zeros(100000)

# Print the number of samples in the training and test datasets
print("Number of samples in the training dataset: ", X_train.shape[0])
print("Number of samples in the test dataset: ", X_test.shape[0])
print("Number of anomalies in the test dataset: ", y_test[y_test == 1].shape[0])
print("Number of normal samples in the test dataset: ", y_test[y_test == 0].shape[0])

Number of samples in the training dataset:  100000
Number of samples in the test dataset:  25000
Number of anomalies in the test dataset:  12500
Number of normal samples in the test dataset:  12500


## Unsupervised learning - Local Outlier Factor (LOF)

In [45]:
from sklearn.preprocessing import MaxAbsScaler, MinMaxScaler, StandardScaler

# Scale the data using MaxAbsScaler
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ | Data scaled using StandardScaler")

✅ | Data scaled using StandardScaler


In [50]:
# Local Outlier Factor (LOF)
from sklearn.neighbors import LocalOutlierFactor
clf = LocalOutlierFactor(novelty=True)
param_grid = {
    'n_neighbors': [2,5,10,15,20],
    'algorithm': ['auto', 'ball_tree', 'kd_tree'],
    'leaf_size': [2, 3, 5],
    'metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=15, verbose=1)
grid_search.fit(X_train_scaled, y_train)

best_params = grid_search.best_params_
print("Best parameters: ", best_params) 

# Train final model with best parameters
clf = LocalOutlierFactor(**best_params, novelty=True)
clf.fit(X_train_scaled)

Fitting 3 folds for each of 90 candidates, totalling 270 fits
Best parameters:  {'algorithm': 'auto', 'leaf_size': 2, 'metric': 'euclidean', 'n_neighbors': 2}


,"n_neighbors n_neighbors: int, default=20Number of neighbors to use by default for :meth:`kneighbors` queries.If n_neighbors is larger than the number of samples provided,all samples will be used.",2
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf is size passed to :class:`BallTree` or :class:`KDTree`. This canaffect the speed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"p p: float, default=2Parameter for the Minkowski metric from:func:`sklearn.metrics.pairwise_distances`. When p = 1, thisis equivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. When fitting this is used to define thethreshold on the scores of the samples.- if 'auto', the threshold is determined as in the original paper,- if a float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"novelty novelty: bool, default=FalseBy default, LocalOutlierFactor is only meant to be used for outlierdetection (novelty=False). Set novelty to True if you want to useLocalOutlierFactor for novelty detection. In this case be aware thatyou should only use predict, decision_function and score_sampleson new unseen data and not on the training set; and note that theresults obtained this way may differ from the standard LOF results... versionadded:: 0.20",True
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [51]:
# Evaluate the classifier
y_pred_raw = clf.predict(X_test_scaled)
y_pred = np.where(y_pred_raw == -1, 1, 0)

# Calculate metrics
tp = np.sum((y_pred == 1) & (y_test == 1))
tn = np.sum((y_pred == 0) & (y_test == 0))
fp = np.sum((y_pred == 1) & (y_test == 0))
fn = np.sum((y_pred == 0) & (y_test == 1))

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
false_positive = fp / (fp + tn) if (fp + tn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)
print("F1 Score: ", f1)
print("False Positive Rate: ", false_positive)
print("TP: ", tp, "TN: ", tn, "FP: ", fp, "FN: ", fn)

Accuracy:  0.79916
Precision:  0.7184670210901443
Recall:  0.98384
F1 Score:  0.8304689874058817
False Positive Rate:  0.38552
TP:  12298 TN:  7681 FP:  4819 FN:  202


## Unsupervised learning - OCSVM

In [31]:
from sklearn.preprocessing import MaxAbsScaler

# Scale the data using MaxAbsScaler
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ | Data scaled using MaxAbsScaler")

✅ | Data scaled using MaxAbsScaler


In [33]:
# One-Class SVM - Grid search for hyperparameter optimization
from sklearn.svm import OneClassSVM
from sklearn.model_selection import GridSearchCV

# Create the One-Class SVM classifier
# Best parameters based: {'coef0': 0.0, 'degree': 2, 'gamma': 'scale', 'kernel': 'rbf', 'nu': 0.005, 'tol': 0.0001}
clf = OneClassSVM(coef0=0.0, degree=2, gamma='scale', kernel='rbf', nu=0.005, tol=0.0001)
clf.fit(X_train_scaled)

# Parameter grid for One-Class SVM
# param_grid = {
#     'kernel': ['rbf'],
#     'gamma': ['scale', 'auto', 0.0001, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0],
#     'nu': [0.005, 0.01, 0.015, 0.02, 0.03, 0.05, 0.1, 0.15],
#     'degree': [2, 3],
#     'coef0': [0.0, 0.1],
#     'tol': [1e-4, 1e-3, 1e-2],
# }

# # Grid search with cross-validation
# grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=15, verbose=1)
# grid_search.fit(X_train_scaled, y_train)
# best_params = grid_search.best_params_
# print("Best parameters: ", best_params)

# # Train final model with best parameters
# clf = OneClassSVM(**best_params)
# clf.fit(X_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",2
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.0001
,"nu nu: float, default=0.5An upper bound on the fraction of trainingerrors and a lower bound of the fraction of supportvectors. Should be in the interval (0, 1]. By default 0.5will be taken.",0.005
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [34]:
# Evaluate the One-Class SVM classifier
# One-Class SVM returns: -1 for anomalies (outliers), 1 for inliers (normal)
# Convert to: 1 for anomalies, 0 for normal (benign)
y_pred_raw = clf.predict(X_test_scaled)
y_pred = np.where(y_pred_raw == -1, 1, 0)  # -1 (anomaly) -> 1, 1 (inlier) -> 0

# Calculate metrics
tp = np.sum((y_pred == 1) & (y_test == 1))
tn = np.sum((y_pred == 0) & (y_test == 0))
fp = np.sum((y_pred == 1) & (y_test == 0))
fn = np.sum((y_pred == 0) & (y_test == 1))

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
false_positive = fp / (fp + tn) if (fp + tn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)
print("F1 Score: ", f1)
print("False Positive Rate: ", false_positive)
print("TP: ", tp, "TN: ", tn, "FP: ", fp, "FN: ", fn)

Accuracy:  0.94596
Precision:  0.9840236172614396
Recall:  0.90664
F1 Score:  0.943748178373652
False Positive Rate:  0.01472
TP:  11333 TN:  12316 FP:  184 FN:  1167


## Unsupervised Online Learning Using One-Class SVM (oSVM)

In [96]:
# Read CSV
dataset = pd.read_csv(DATASET_PATH_DATASET2 + DATASET_NAME_DATASET2, sep=',', index_col=False)

# Features
FEATURES = ['IPV_SRC_ADDR', 'IPV_DST_ADDR', 'L_DST_PORT', 'L_SRC_PORT', 'DIRECTION', 'FIRST_SWITCHED', 'LAST_SWITCHED', 'PROTOCOL', 'END_TYPE', 'IN_BYTES', 'OUT_BYTES', 'Label']

# Preprocess flows
dataset = dataset[FEATURES]
dataset = dataset.iloc[:, :].values

In [97]:
# Significative port
print("ℹ️ | MODE SIGNIFICANT PORT ACTIVATED")
for i in range(len(dataset)):
    if dataset[i][FEATURES.index('DIRECTION')] == 1:
        dataset[i][FEATURES.index('DIRECTION')] = dataset[i][FEATURES.index('L_DST_PORT')]
    else:
        dataset[i][FEATURES.index('DIRECTION')] = dataset[i][FEATURES.index('L_SRC_PORT')]
print('✅ | Significant port added')
    

for i in range(len(dataset)):
    dataset[i][FEATURES.index('FIRST_SWITCHED')] = int(dataset[i][FEATURES.index('FIRST_SWITCHED')].split(' ')[1].split(':')[0])

ℹ️ | MODE SIGNIFICANT PORT ACTIVATED
✅ | Significant port added


In [98]:
# Transform numpy array to pandas dataframe
dataset = pd.DataFrame(dataset, columns=FEATURES)

# Significant port
dataset = dataset.drop(columns=['L_SRC_PORT', 'L_DST_PORT'])
# Change DIRECTION column name to PORT
dataset = dataset.rename(columns={'DIRECTION': 'PORT'})



# Create the two datasets for trainning
datasets_training = dataset.loc[dataset['Label'] == 0].iloc[:FLOWS_TRAIN_SCALER + FLOWS_TRAIN_OML]
dataset_train_scaler = datasets_training.loc[dataset['Label'] == 0].iloc[:FLOWS_TRAIN_SCALER]
dataset_train_oml = datasets_training.loc[dataset['Label'] == 0].iloc[FLOWS_TRAIN_SCALER:FLOWS_TRAIN_SCALER + FLOWS_TRAIN_OML + 1]


# Drop train_scaler and train_oml from the dataset
dataset = dataset.drop(dataset_train_scaler.index)
dataset = dataset.drop(dataset_train_oml.index)


# Number of anomalies and normal samples
anomalies_samples= dataset.loc[dataset['Label'] == 1].shape[0]
benign_samples = dataset.loc[dataset['Label'] == 0].shape[0]

# Drop benign samples to equilibrate the dataset
if benign_samples > anomalies_samples:
    dataset = dataset.drop(dataset.loc[dataset['Label'] == 0].index[:benign_samples - anomalies_samples])
else:
    dataset = dataset.drop(dataset.loc[dataset['Label'] == 1].index[:anomalies_samples- benign_samples])

# Create the test dataset
dataset_test_labeled = dataset

# Shuffle the datasets
dataset_train_scaler = dataset_train_scaler.sample(frac=1,random_state=111).reset_index(drop=True)
dataset_train_oml = dataset_train_oml.sample(frac=1,random_state=111).reset_index(drop=True)
dataset_test_labeled = dataset_test_labeled.sample(frac=1,random_state=111).reset_index(drop=True)

In [99]:
encoder = preprocessing.OrdinalEncoder()

for i in range(len(dataset_train_scaler)):
    # Obtain first two elements (IP addresses) pandas datraframe
    first_two_elements = {j: str(dataset_train_scaler.iloc[i, j]) for j in range(2)}
    # Encode the IP addresses
    encoder.learn_one(first_two_elements)
    first_two_elements = encoder.transform_one(first_two_elements)
    dataset_train_scaler.iloc[i, 0] = first_two_elements[0]
    dataset_train_scaler.iloc[i, 1] = first_two_elements[1]

for i in range(len(dataset_train_oml)):
    # Obtain first two elements (IP addresses)
    first_two_elements = {j: str(dataset_train_oml.iloc[i, j]) for j in range(2)}
    # Encode the IP addresses
    encoder.learn_one(first_two_elements)
    first_two_elements = encoder.transform_one(first_two_elements)
    dataset_train_oml.iloc[i, 0] = first_two_elements[0]
    dataset_train_oml.iloc[i, 1] = first_two_elements[1]

for i in range(len(dataset_test_labeled)):
    # Obtain first two elements (IP addresses)
    first_two_elements = {j: str(dataset_test_labeled.iloc[i, j]) for j in range(2)}
    # Encode the IP addresses
    encoder.learn_one(first_two_elements)
    first_two_elements = encoder.transform_one(first_two_elements)
    dataset_test_labeled.iloc[i, 0] = first_two_elements[0]
    dataset_test_labeled.iloc[i, 1] = first_two_elements[1]

In [100]:
print('=====================================================================================================')
print('Number of anomalies in the train scaler dataset:\t', dataset_train_scaler.loc[dataset_train_scaler['Label'] == 1].shape[0])
print('Number of normal samples in the train scaler dataset:\t', dataset_train_scaler.loc[dataset_train_scaler['Label'] == 0].shape[0])

print('Number of normal samples in the trainOML dataset:\t', dataset_train_oml.loc[dataset_train_oml['Label'] == 0].shape[0])
print('Number of anomaly samples in the trainOML dataset:\t', dataset_train_oml.loc[dataset_train_oml['Label'] == 1].shape[0])

print('Number of anomalies in the test dataset:\t', dataset_test_labeled.loc[dataset_test_labeled['Label'] == 1].shape[0])
print('Number of normal samples in the test dataset:\t', dataset_test_labeled.loc[dataset_test_labeled['Label'] == 0].shape[0])
print('=====================================================================================================')

# Obtain the datasets without the label
dataset_train_no_labels = np.delete(dataset_train_oml, -1, axis=1)
dataset_test_no_labels = np.delete(dataset_test_labeled, -1, axis=1)
dataset_train_scaler = dataset_train_scaler.drop(columns=['Label'])

Number of anomalies in the train scaler dataset:	 0
Number of normal samples in the train scaler dataset:	 1000
Number of normal samples in the trainOML dataset:	 100000
Number of anomaly samples in the trainOML dataset:	 0
Number of anomalies in the test dataset:	 12500
Number of normal samples in the test dataset:	 12500


In [101]:
scaler = preprocessing.MaxAbsScaler()

scaler_dataset_train = []
scaler_dataset_test = []

# Convert the dataset to pandas DataFrame format
dataset_train_scaler = pd.DataFrame(dataset_train_scaler)
dataset_train_no_labels = pd.DataFrame(dataset_train_no_labels)
dataset_test_no_labels = pd.DataFrame(dataset_test_no_labels)

# Train the scaler model using the training dataset
for _, row in dataset_train_scaler.iterrows():
    # Convert row to dict using keys 0, 1, 2, 3, ...
    row = {i: value for i, value in enumerate(row)}
    scaler.learn_one(row)

# Scale the training dataset using the trained scaler
for _, row in dataset_train_no_labels.iterrows():
    row = row.to_dict()
    row = scaler.transform_one(row)
    scaler_dataset_train.append(list(row.values()))

# Scale the test dataset using the previously trained scaler
for _, row in dataset_test_no_labels.iterrows():
    row = row.to_dict()
    row = scaler.transform_one(row)
    scaler_dataset_test.append(list(row.values()))

print("✅ | Dataset Scaled")


✅ | Dataset Scaled


In [102]:
model = anomaly.QuantileFilter(
        anomaly.OneClassSVM(nu=0.05,intercept_lr=optim.schedulers.InverseScaling(learning_rate=0.25)),
        q = 0.99
    )

probability_preprocessing = preprocessing.MinMaxScaler()
probability = sketch.Histogram()


In [103]:
# Traning phase of OML
print("⏳ | Training OML")
for row in scaler_dataset_train:
    # Change dict to numpy array
    row_dict = {f'feature_{i}': value for i, value in enumerate(row)}

    model.learn_one(row_dict)

    score = model.score_one(row_dict)
    probability_preprocessing.learn_one({0: score})
    probability.update(probability_preprocessing.transform_one({0: score})[0])

print("✅ | Training phase completed")

⏳ | Training OML
✅ | Training phase completed


In [104]:
fp = 0
fn = 0
tp = 0
tn = 0
accuracies = []
recalls = []
false_positives = []
anomalies = []



for i, row in enumerate(scaler_dataset_test):
    # To dict
    if isinstance(row, list):
        row = {f'feature_{j}': value for j, value in enumerate(row)}

    score = model.score_one(row)
    anomalo = model.classify(score)
    anomalies.append(anomalo)

    # Update probability
    probability_preprocessing.learn_one({0: score})
    probability.update(probability_preprocessing.transform_one({0: score})[0])
    rank = probability.cdf(probability_preprocessing.transform_one({0: score})[0])


    if not anomalo:
        model.learn_one(row)

    label = dataset_test_labeled.iloc[i, -1]

    if anomalo and label == 1:
        tp += 1
    elif not anomalo and label == 0:
        tn += 1
    elif anomalo and label == 0:
        fp += 1
    elif not anomalo and label == 1:
        fn += 1
        
    accuracies.append((tp + tn) / (tp + tn + fp + fn))
    recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    false_positives.append(fp / (fp + tn) if (fp + tn) > 0 else 0)


accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
false_positive = fp / (fp + tn) if (fp + tn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)
print("F1 Score: ", f1)
print("False Positive Rate: ", false_positive)
print("TP: ", tp, "TN: ", tn, "FP: ", fp, "FN: ", fn)

Accuracy:  0.98712
Precision:  0.9881353214686548
Recall:  0.98608
F1 Score:  0.9871065908544886
False Positive Rate:  0.01184
TP:  12326 TN:  12352 FP:  148 FN:  174


The best results were achieved by this model, which is capable of detecting anomalies without relying on a labeled dataset—in other words, by analyzing the underlying patterns in the data.